In [1]:
import pandas as pd
import pandas as pd
from win32com.client import Dispatch
from win32com.client import DispatchEx


In [2]:
def get_sheet(workbook, target_name):
    target = target_name.strip().lower()

    for ws in workbook.Worksheets:
        name = ws.Name.strip().lower()
        if name == target:
            return ws

    raise Exception(f"Feuille '{target_name}' introuvable")

In [3]:
def get_or_open(excel, path):
    for wb in excel.Workbooks:
        if wb.FullName.lower() == path.lower():
            return wb
    return excel.Workbooks.Open(path)

In [4]:
path_fichier_vl_n = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\Ficheir VL n.xlsx"
path_fichier_vl_n_1 = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\Fichier VL n_1.xlsx"
path_comptes_annuells = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Old fund\2025.12. Sancy  .Comptes annuels.xlsm"

# Extracting Data

## Extracting Balance, Inventaire, PVMV N

In [5]:
balance_n = pd.read_excel(path_fichier_vl_n, "Balance ", header=None, dtype=str)
inventaire_n = pd.read_excel(path_fichier_vl_n, "Inventaire", header=None, dtype=str)
pvmv = pd.read_excel(path_fichier_vl_n, "PVMV", header=None, dtype=str)

## Extracting Balance, Inventaire N-1

In [6]:
balance_n_1 = pd.read_excel(path_fichier_vl_n_1, "Balance ", header=None, dtype=str)
inventaire_n_1 = pd.read_excel(path_fichier_vl_n_1, "Inventaire", header=None, dtype=str)

# Extraction des donnees depuis PDF

In [11]:
path_pdf_file = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Singular Ventures I - Comptes annuels 31.12.2025 v3.pdf"

In [18]:
import pdfplumber as pdf
import pandas as pd
def read_pdf_file(path_to_pdf_file):
    """
    Returns a list containing the text of each page.
    """
    tables = []
    texts = []
    with pdf.open(path_to_pdf_file) as p:
        i = 0
        for page in p.pages:
           texts.append({i : page.extract_text() or ""} )
           tables.append({i : page.extract_table() or "" }) 
           i+=1
    
    return texts, tables

In [19]:
texts, tables = read_pdf_file(path_to_pdf_file=path_pdf_file)

In [23]:
"""
pdf_to_dict.py
==============
Extrait un dictionnaire structuré depuis un PDF de comptes annuels FPCI Aplitec.

Les clés sont STATIQUES et correspondent exactement aux noms de sheets du fichier
Excel Coala (le modèle source du rapport) :

    "Bilan Actif"              → tableau N / N-1
    "Bilan Passif"             → tableau N / N-1
    "Compte de résultat 1"     → tableau N / N-1
    "Compte de résultat 2"     → tableau N / N-1
    "I. Prospectus"            → {titre: texte}
    "I. Caractéristiques"      → tableau 5 exercices
    "II. Copie règlement 1"    → {titre: texte, titre: texte, ...}
    "II. Copie règlement 2"    → {titre: texte, ...}
    "III. Capitaux propres"    → tableau 3 colonnes
    "IV. Evol parts"           → tableaux souscrits/rachetés
    "V. Flux nominal"          → tableau par parts
    "VI. Ventilation AN 1"     → texte méthode
    "VI. Ventilation AN 2"     → tableau 6.2
    "VIII. Exposition portefeuille" → tableau 8.1
    "Cessions"                 → tableau 8.4
    "Créances et frais"        → tableaux IX + X
    "Hors-bilan et autres infos" → tableaux XI + XII
    "XIII. Sommes distribuables" → tableaux XIII
    "XIV. Elements de bilan"   → inventaire instruments

Usage :
    from pdf_to_dict import extract
    d = extract("comptes_annuels.pdf")
    print(d["Bilan Actif"])
"""

import re
from collections import defaultdict
import pdfplumber


# ══════════════════════════════════════════════════════════════
#  UTILITAIRES
# ══════════════════════════════════════════════════════════════

def _clean(v):
    if v is None:
        return ""
    return re.sub(r"\s+", " ", str(v).replace("\xa0", " ")).strip()

def _num(s):
    """Convertit une chaîne en float. Retourne None si impossible."""
    if s is None:
        return None
    s = _clean(s).replace(" ", "").replace("\u202f", "")
    neg = s.startswith("-") or s.startswith("−")
    s = s.lstrip("-−").replace(",", ".")
    try:
        v = float(s)
        return -v if neg else v
    except ValueError:
        return None

def _is_dash(s):
    return _clean(s) in ("-", "–", "−", "")

def _num_or_none(s):
    if _is_dash(str(s)):
        return None
    return _num(s)


# ══════════════════════════════════════════════════════════════
#  EXTRACTION SPATIALE — cœur du module
# ══════════════════════════════════════════════════════════════

def _words_to_rows(words, y_tol=3):
    """Regroupe les mots d'une page en lignes par coordonnée Y."""
    buckets = defaultdict(list)
    for w in words:
        y = round(w["top"] / y_tol) * y_tol
        buckets[y].append(w)
    return {y: sorted(ws, key=lambda w: w["x0"]) for y, ws in sorted(buckets.items())}


def _row_cols(row_words, col_N=(360, 455), col_N1=(455, 545)):
    """
    Sépare une ligne de mots en : libellé | valeur_N | valeur_N1.
    Retourne (label, val_N_str, val_N1_str).
    """
    label_parts, n_parts, n1_parts = [], [], []
    for w in row_words:
        x = w["x0"]
        t = w["text"]
        if col_N[0] <= x < col_N[1]:
            n_parts.append(t)
        elif col_N1[0] <= x < col_N1[1]:
            n1_parts.append(t)
        else:
            label_parts.append(t)
    label = " ".join(label_parts).strip()
    val_n = " ".join(n_parts).strip()
    val_n1 = " ".join(n1_parts).strip()
    return label, val_n, val_n1


def _extract_two_col_table(pages_idx, pdf, col_N=(360, 455), col_N1=(455, 545)):
    """
    Extrait un tableau N/N-1 sur les pages indiquées.
    Retourne une liste de {"libelle", "niveau", "N", "N1"}.
    """
    rows = []
    for pi in pages_idx:
        page = pdf.pages[pi]
        words = page.extract_words()
        by_y = _words_to_rows(words)
        for y, row_words in by_y.items():
            label, val_n, val_n1 = _row_cols(row_words, col_N, col_N1)
            if not label:
                continue
            # Calculer le niveau d'indentation par la position x du premier mot
            niveau = 0
            if row_words:
                x0 = row_words[0]["x0"]
                if x0 > 140:
                    niveau = 2
                elif x0 > 90:
                    niveau = 1
            rows.append({
                "libelle": label,
                "niveau": niveau,
                "N": _num_or_none(val_n),
                "N1": _num_or_none(val_n1),
            })
    return rows


def _extract_text_pages(pages_idx, pdf):
    """Retourne le texte brut concaténé des pages."""
    return "\n".join(pdf.pages[i].extract_text() or "" for i in pages_idx)


def _split_multirow_table(page):
    """
    Extrait une table PDF où chaque cellule contient plusieurs lignes (\n).
    Retourne une liste de listes (une par instrument).
    """
    result = []
    tables = page.extract_tables()
    for tbl in tables:
        if not tbl or len(tbl) < 2:
            continue
        # Chercher la ligne d'en-tête réels
        header_idx = None
        for i, row in enumerate(tbl):
            if row and any("Clôture" in str(c) or "Quantité" in str(c)
                           or "Prix de cession" in str(c) for c in row if c):
                header_idx = i
                break
        if header_idx is None:
            continue
        for row in tbl[header_idx + 1:]:
            if not row or not row[0]:
                continue
            # Chaque cellule peut contenir plusieurs valeurs séparées par \n
            cols = [str(c).split("\n") if c else [""] for c in row]
            max_len = max(len(c) for c in cols)
            cols_padded = [c + [""] * (max_len - len(c)) for c in cols]
            for values in zip(*cols_padded):
                nom = values[0].strip()
                if not nom or nom in ("Sous Total", "Total"):
                    result.append({"_total": True, "values": list(values)})
                    continue
                result.append({"nom": nom, "values": list(values[1:])})
    return result


# ══════════════════════════════════════════════════════════════
#  PARSEURS PAR SECTION
# ══════════════════════════════════════════════════════════════

# ── Bilan Actif (page 2) ────────────────────────────────────
def _bilan_actif(pdf):
    """
    Clé: "Bilan Actif"
    Structure: liste ordonnée de lignes avec libelle, niveau, N, N1
    """
    rows = _extract_two_col_table([1], pdf)
    # Filtrer les lignes de titres de page et footnotes
    filtered = [
        r for r in rows
        if r["libelle"] not in ("Bilan Actif au 31/12/2025 en euros Exercice N Exercice N-1",
                                 "Exercice N Exercice N-1")
        and not r["libelle"].startswith("(1) Les autres")
        and not r["libelle"].startswith("SINGULAR")
        and not r["libelle"].startswith("Page ")
    ]
    return filtered


# ── Bilan Passif (page 3) ───────────────────────────────────
def _bilan_passif(pdf):
    """Clé: "Bilan Passif" """
    rows = _extract_two_col_table([2], pdf)
    filtered = [
        r for r in rows
        if not r["libelle"].startswith("Bilan Passif")
        and not r["libelle"].startswith("SINGULAR")
        and not r["libelle"].startswith("Page ")
    ]
    return filtered


# ── Compte de résultat 1 (page 4) ──────────────────────────
def _compte_resultat_1(pdf):
    """Clé: "Compte de résultat 1" """
    rows = _extract_two_col_table([3], pdf)
    filtered = [
        r for r in rows
        if not r["libelle"].startswith("Compte de résultat")
        and not r["libelle"].startswith("SINGULAR")
        and not r["libelle"].startswith("Page ")
        and not r["libelle"].startswith("(1) Seulement")
    ]
    return filtered


# ── Compte de résultat 2 (page 5) ──────────────────────────
def _compte_resultat_2(pdf):
    """Clé: "Compte de résultat 2" """
    rows = _extract_two_col_table([4], pdf)
    filtered = [
        r for r in rows
        if not r["libelle"].startswith("Compte de résultat")
        and not r["libelle"].startswith("SINGULAR")
        and not r["libelle"].startswith("Page ")
    ]
    return filtered


# ── I. Prospectus (page 6) ──────────────────────────────────
def _prospectus(pdf):
    """
    Clé: "I. Prospectus"
    Structure: {
        "1.1 - Informations relatives à la stratégie de l'OPC ainsi que les stratégies de gestion du risque afférentes": "texte..."
    }
    """
    text = _extract_text_pages([5], pdf)
    # Supprimer l'en-tête et pied de page
    text = re.sub(r"SINGULAR VENTURES.*?\n", "", text)
    text = re.sub(r"ANNEXE AUX COMPTES ANNUELS\n?", "", text)
    text = re.sub(r"I\. Caractéristiques et activité de l'OPC\s*\n?", "", text)
    text = re.sub(r"Page \d+ sur \d+\s*", "", text)

    titre = "1.1 - Informations relatives à la stratégie de l'OPC ainsi que les stratégies de gestion du risque afférentes"
    # Extraire le texte après le titre
    m = re.search(r"1\.1[^\n]*\n(.*)", text, re.DOTALL)
    contenu = _clean(m.group(1)) if m else _clean(text)

    return {titre: contenu}


# ── I. Caractéristiques (page 7) ────────────────────────────
def _caracteristiques(pdf):
    """
    Clé: "I. Caractéristiques"
    Structure: {
        "1.2 - Eléments caractéristiques de l'OPC au cours des cinq derniers exercices en euros": {
            "dates": ["31/12/2025", ...],
            "actif_net": {"31/12/2025": val, ...},
            "Parts A": {"engagement_souscription": {...}, "montant_libere": {...},
                        "nombre_parts": {...}, "valeur_liquidative": {...}},
            "Parts A'": {...},
            "Parts B": {...}
        }
    }
    """
    titre = "1.2 - Eléments caractéristiques de l'OPC au cours des cinq derniers exercices en euros"
    result = {
        titre: {
            "dates": [],
            "actif_net": {},
            "Parts A": {},
            "Parts A'": {},
            "Parts B": {},
        }
    }
    inner = result[titre]

    text = _extract_text_pages([6], pdf)
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    # Trouver les dates (ligne "Libellé 31/12/2025 31/12/2024 ...")
    dates = []
    for line in lines:
        m = re.findall(r"31/12/\d{4}\*{0,2}", line)
        if len(m) >= 2:
            dates = m
            break
    inner["dates"] = dates

    if not dates:
        return result

    def _parse_5_vals(line, label):
        """Extrait jusqu'à 5 valeurs numériques d'une ligne en retirant le label."""
        s = line.replace(label, "").strip()
        tokens = s.split()
        vals = []
        i = 0
        while i < len(tokens) and len(vals) < 5:
            # Reconstituir les nombres avec espaces (milliers)
            num_str = tokens[i]
            # Vérifier si c'est un début de nombre
            if re.match(r"^\d", num_str) or re.match(r"^-\d", num_str):
                # Accumuler les tokens suivants s'ils sont des groupes de 3 chiffres
                while i + 1 < len(tokens) and re.match(r"^\d{3}$", tokens[i + 1]):
                    num_str += tokens[i + 1]
                    i += 1
                vals.append(_num(num_str))
            elif num_str in ("-", "–"):
                vals.append(None)
            i += 1
        return {dates[j]: vals[j] for j in range(min(len(dates), len(vals)))}

    # Parser via table pdfplumber (cellules multirow)
    page = pdf.pages[6]
    tables = page.extract_tables()

    # L'ordre exact des champs dans la cellule label (après le nom de la part)
    FIELDS_ORDER = [
        "engagement_souscription",    # Engagement de souscription
        "montant_libere",             # Montant libéré
        "repartition_avoirs",         # Répartition d'avoirs*
        "nombre_parts",               # Nombre de parts
        "distribution_revenu_net",    # Distribution unitaire sur revenu net
        "distribution_pmv_realisees", # Distribution unitaire sur plus et moins-values réalisées nettes
        "valeur_liquidative",         # Valeur liquidative (dernière ligne)
    ]

    # Mapping du nombre de valeurs par part (certains champs peuvent être absents ou vides)
    # On doit compter les lignes réelles dans la cellule label
    for tbl in tables:
        if not tbl or len(tbl) < 2:
            continue
        # Ligne 0 = en-têtes dates
        header_row = tbl[0]
        hdr_dates = [_clean(str(c)) for c in header_row[1:] if c]
        if len(hdr_dates) >= 2:
            dates = hdr_dates
            inner["dates"] = dates

        # Ligne 1 = Actif net
        if len(tbl) > 1:
            row_an = tbl[1]
            vals = [_num_or_none(_clean(str(c))) for c in row_an[1:] if c]
            if vals and dates:
                inner["actif_net"] = {dates[i]: vals[i] for i in range(min(len(dates), len(vals)))}

        # Lignes 2..4 = Parts A, A', B
        for row in tbl[2:]:
            if not row or not row[0]:
                continue
            cell0 = str(row[0])
            lines_c0 = [l.strip() for l in cell0.split("\n") if l.strip()]
            if not lines_c0:
                continue

            # Identifier la catégorie
            part_name = None
            for p in ("Parts A'", "Parts A", "Parts B"):
                if lines_c0[0] == p or lines_c0[0].startswith(p):
                    part_name = p
                    break
            if not part_name:
                continue

            # Les labels de champs sont dans lines_c0[1:]
            field_labels = lines_c0[1:]

            # Pour chaque colonne (date), récupérer les valeurs ligne par ligne
            # La valeur de chaque champ correspond à sa position dans field_labels
            for col_idx, date in enumerate(dates):
                if col_idx + 1 >= len(row):
                    break
                cell_val = str(row[col_idx + 1]) if row[col_idx + 1] else ""
                vals_in_col = [v.strip() for v in cell_val.split("\n") if v.strip() != ""]

                # Associer position par position avec field_labels
                # IMPORTANT : les champs "Distribution unitaire" sont vides dans les colonnes
                # (le PDF ne les inclut pas dans les valeurs) → on les skip
                _EMPTY_FIELDS = {
                    "distribution_revenu_net",
                    "distribution_pmv_realisees",
                    "distribution_pmv_latentes",
                }

                col_pos = 0  # position courante dans vals_in_col
                for field_idx, field_label in enumerate(field_labels):
                    # Retrouver la clé
                    field_key = None
                    if "Engagement de souscription" in field_label:
                        field_key = "engagement_souscription"
                    elif "Montant libéré" in field_label:
                        field_key = "montant_libere"
                    elif "Répartition d'avoirs" in field_label:
                        field_key = "repartition_avoirs"
                    elif "Nombre de parts" in field_label:
                        field_key = "nombre_parts"
                    elif "Distribution unitaire sur revenu net" == field_label:
                        field_key = "distribution_revenu_net"
                    elif "Distribution unitaire sur plus et" in field_label and "réalisées" in field_label:
                        field_key = "distribution_pmv_realisees"
                    elif "Distribution unitaire sur plus et" in field_label and "latentes" in field_label:
                        field_key = "distribution_pmv_latentes"
                    elif "Valeur liquidative" in field_label:
                        field_key = "valeur_liquidative"

                    if not field_key:
                        continue

                    # Les champs Distribution sont vides → pas de valeur dans la colonne
                    if field_key in _EMPTY_FIELDS:
                        # Ne pas avancer col_pos
                        inner[part_name].setdefault(field_key, {})[date] = None
                        continue

                    val_str = vals_in_col[col_pos] if col_pos < len(vals_in_col) else ""
                    val = _num_or_none(val_str)
                    inner[part_name].setdefault(field_key, {})[date] = val
                    col_pos += 1

    return result


# ── II. Copie règlement 1 (page 8) ──────────────────────────
def _regles_1(pdf):
    """
    Clé: "II. Copie règlement 1"
    Structure: {
        "2.1 - Règles et méthodes comptables appliquées au cours de l'exercice": "texte...",
        "2.2 - Description des méthodes de valorisation des postes de bilan": "texte...",
        "Instruments financiers non négociés sur un marché réglementé ou assimilé": "texte..."
    }
    """
    text = _extract_text_pages([7], pdf)
    text = re.sub(r"SINGULAR VENTURES[^\n]*\n?", "", text)
    text = re.sub(r"Page \d+ sur \d+\s*", "", text)
    text = re.sub(r"II\. Règles et méthodes comptables\s*\n?", "", text)

    result = {}

    # 2.1
    m = re.search(r"(2\.1[^\n]*)\n(.*?)(?=2\.2)", text, re.DOTALL)
    if m:
        result[_clean(m.group(1))] = _clean(m.group(2))

    # 2.2 + sous-sections
    m = re.search(r"(2\.2[^\n]*)\n(.*?)(?=Instruments financiers n[eé]goci[eé]s sur un march[eé] r[eé]glement|$)", text, re.DOTALL)
    if m:
        result[_clean(m.group(1))] = _clean(m.group(2))

    # Sous-section instruments non cotés (dans 2.2)
    m = re.search(r"(Instruments financiers non n[eé]goci[eé]s[^\n]*)\n?(.*?)$", text, re.DOTALL)
    if m:
        result[_clean(m.group(1))] = _clean(m.group(2))

    return result


# ── II. Copie règlement 2 (page 9) ──────────────────────────
def _regles_2(pdf):
    """
    Clé: "II. Copie règlement 2"
    Structure: {
        "Instruments financiers négociés sur un marché réglementé ou assimilé": "texte...",
        "Parts ou actions d'OPC et droits d'entrée d'investissement": "texte...",
        "Instruments financiers à terme (prêts)": "texte..."
    }
    """
    text = _extract_text_pages([8], pdf)
    text = re.sub(r"SINGULAR VENTURES[^\n]*\n?", "", text)
    text = re.sub(r"Page \d+ sur \d+\s*", "", text)

    result = {}
    sections = [
        "Instruments financiers négociés sur un marché réglementé ou assimilé",
        "Parts ou actions d'OPC et droits d'entrée d'investissement",
        "Instruments financiers à terme (prêts)",
    ]

    for i, sec in enumerate(sections):
        pattern = re.escape(sec) + r"\s*\n?(.*?)(?=" + (
            re.escape(sections[i + 1]) if i + 1 < len(sections) else r"$"
        ) + ")"
        m = re.search(pattern, text, re.DOTALL)
        result[sec] = _clean(m.group(1)) if m else ""

    return result


# ── III. Capitaux propres (page 10) ─────────────────────────
def _capitaux_propres(pdf):
    """
    Clé: "III. Capitaux propres"
    Structure: {
        "III. Reconstitution de la ligne \"capitaux propres\"": {
            "lignes": [{"libelle", "signe", "cumul_N", "cumul_N1", "variation_N"}, ...],
            "notes": {"frais_constitution": val, "prime_souscription": val}
        }
    }
    """
    titre = 'III. Reconstitution de la ligne "capitaux propres"'
    result = {titre: {"lignes": [], "notes": {}}}
    inner = result[titre]

    with pdfplumber.open(pdf._fp.name if hasattr(pdf, "_fp") else pdf) as _pdf:
        page = _pdf.pages[9]

    words = page.extract_words()
    by_y = _words_to_rows(words)

    # 3 colonnes : Cumul N (~330-450), Cumul N-1 (~450-520), Variation (~520-580)
    col_cumN  = (325, 450)
    col_cumN1 = (450, 525)
    col_var   = (525, 590)

    for y, row_words in by_y.items():
        label_parts, cumN_parts, cumN1_parts, var_parts, signe = [], [], [], [], None

        for w in row_words:
            x = w["x0"]
            t = w["text"]
            if t in ("+", "-", "+/-", "="):
                signe = t
            elif col_cumN[0] <= x < col_cumN[1]:
                cumN_parts.append(t)
            elif col_cumN1[0] <= x < col_cumN1[1]:
                cumN1_parts.append(t)
            elif col_var[0] <= x < col_var[1]:
                var_parts.append(t)
            else:
                if t not in ("+", "-", "+/-", "="):
                    label_parts.append(t)

        label = " ".join(label_parts).strip()
        if not label or label.startswith("SINGULAR") or label.startswith("Page") \
                or label.startswith("III. Recon") or "depuis" in label \
                or label.startswith("(") or label in ("Cumul Exercice N", "Cumul Exercice N-1",
                                                        "Variation Exercice N", "Reconstitution"):
            continue

        # Notes de bas de page
        if "Frais de constitution" in label:
            m = re.search(r"([-\d\s]+)\s*€", label)
            if m:
                inner["notes"]["frais_constitution"] = _num(m.group(1))
            continue
        if "Prime de souscription" in label:
            m = re.search(r"([\d\s]+)\s*€", label)
            if m:
                inner["notes"]["prime_souscription"] = _num(m.group(1))
            continue

        inner["lignes"].append({
            "libelle": label,
            "signe": signe,
            "cumul_N": _num_or_none(" ".join(cumN_parts)),
            "cumul_N1": _num_or_none(" ".join(cumN1_parts)),
            "variation_N": _num_or_none(" ".join(var_parts)),
        })

    return result


# ── IV. Evol parts (page 11) ────────────────────────────────
def _evol_parts(pdf):
    """
    Clé: "IV. Evol parts"
    Structure: {
        "souscrits": [{"type_parts", "nombre", "valeur"}, ...],
        "rachetes":  [{"type_parts", "nombre", "valeur"}, ...],
        "4.2 - Commission de souscription/rachat acquise ou rétrocédée": "Néant."
    }
    Les valeurs sont None si tirets dans le PDF (aucune souscription/rachat).
    """
    result = {
        "souscrits": [],
        "rachetes": [],
        "4.2 - Commission de souscription/rachat acquise ou rétrocédée": "Néant.",
    }
    text = _extract_text_pages([10], pdf)
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    mode = None
    for line in lines:
        if "Souscrits pendant" in line:
            mode = "souscrits"; continue
        if "Rachetés pendant" in line or "Rachetes" in line:
            mode = "rachetes"; continue
        if "4.2" in line and "Commission" in line:
            mode = "commission"; continue
        if mode == "commission" and line:
            result["4.2 - Commission de souscription/rachat acquise ou rétrocédée"] = line
            mode = None; continue

        if mode in ("souscrits", "rachetes"):
            # Les lignes de données : "Parts A - -", "Parts A' - -", etc.
            for part in ("Parts A'", "Parts A", "Parts B", "Total"):
                if line.startswith(part):
                    rest = line[len(part):].strip()
                    # Parser les 2 valeurs (nombre et valeur) — tirets = None
                    tokens = rest.split()
                    def _parse_token(t):
                        if t in ("-", "–"):
                            return None
                        return _num(t)
                    result[mode].append({
                        "type_parts": part,
                        "nombre": _parse_token(tokens[0]) if len(tokens) > 0 else None,
                        "valeur": _parse_token(tokens[1]) if len(tokens) > 1 else None,
                    })
                    break

    return result


# ── V. Flux nominal (page 12) ───────────────────────────────
def _flux_nominal(pdf):
    """
    Clé: "V. Flux nominal"
    Structure: {
        "lignes": [{"libelle", "Parts A", "Parts A'", "Parts B"}, ...],
        "appels_detail": [{"reference", "date", "Parts A", "Parts A'", "Parts B"}, ...]
    }
    Positions spatiales exactes issues du PDF :
        Parts A  x ≈ 300-370
        Parts A' x ≈ 385-445
        Parts B  x ≈ 470-520
    """
    result = {"lignes": [], "appels_detail": []}

    page = pdf.pages[11]
    words = page.extract_words()
    by_y = _words_to_rows(words)

    # Positions connues (vérifiées sur ce PDF)
    col_A  = (295, 375)
    col_Ap = (380, 460)
    col_B  = (465, 525)

    _SKIP_LABELS = {
        "V. Flux concernant le nominal appelé et remboursé sur l'exercice",
        "Exercice N", "Parts A", "Parts A'", "Parts B",
    }

    for y, row_words in by_y.items():
        label_parts, a_parts, ap_parts, b_parts = [], [], [], []
        date_str = None

        for w in row_words:
            x, t = w["x0"], w["text"]
            # Date (format dd/mm/yyyy)
            if re.match(r"\d{2}/\d{2}/\d{4}", t):
                date_str = t
                continue
            if col_A[0] <= x < col_A[1]:
                a_parts.append(t)
            elif col_Ap[0] <= x < col_Ap[1]:
                ap_parts.append(t)
            elif col_B[0] <= x < col_B[1]:
                b_parts.append(t)
            else:
                label_parts.append(t)

        label = " ".join(label_parts).strip()
        if not label or label.startswith("SINGULAR") or label.startswith("Page"):
            continue
        if label in _SKIP_LABELS or label in ("N", "Exercice"):
            continue

        a_val  = _num_or_none(" ".join(a_parts))
        ap_val = _num_or_none(" ".join(ap_parts))
        b_val  = _num_or_none(" ".join(b_parts))
        # Vérifier si la ligne a des tokens dans les colonnes (même des tirets)
        has_col_content = bool(a_parts or ap_parts or b_parts)

        # Appels détaillés
        m = re.match(r"(ADF\s*n[°o][\d]+)", label)
        if m:
            result["appels_detail"].append({
                "reference": m.group(1),
                "date": date_str,
                "Parts A": a_val,
                "Parts A'": ap_val,
                "Parts B": b_val,
            })
        elif has_col_content:
            result["lignes"].append({
                "libelle": label,
                "Parts A": a_val,
                "Parts A'": ap_val,
                "Parts B": b_val,
            })

    return result


# ── VI. Ventilation AN 1 (page 13) ──────────────────────────
def _ventilation_AN1(pdf):
    """
    Clé: "VI. Ventilation AN 1"
    Structure: {
        "6.1 - Description de la méthode de calcul des différentes parts": "texte..."
    }
    """
    titre = "6.1 - Description de la méthode de calcul des différentes parts"
    text = _extract_text_pages([12], pdf)
    text = re.sub(r"SINGULAR VENTURES[^\n]*\n?", "", text)
    text = re.sub(r"Page \d+ sur \d+\s*", "", text)
    text = re.sub(r"VI\. Ventilation de l'actif net par nature de parts\s*\n?", "", text)
    m = re.search(r"6\.1[^\n]*\n(.*)", text, re.DOTALL)
    contenu = _clean(m.group(1)) if m else _clean(text)
    return {titre: contenu}


# ── VI. Ventilation AN 2 (page 14) ──────────────────────────
def _ventilation_AN2(pdf):
    """
    Clé: "VI. Ventilation AN 2"
    Structure: {
        "6.2 - Ventilation de l'actif net par nature de parts": [
            {"type_parts", "nombre_parts", "remboursement_montant_libere",
             "affectation_revenu_prioritaire", "actif_net", "valeur_liquidative"},
            ...
        ]
    }
    """
    titre = "6.2 - Ventilation de l'actif net par nature de parts"
    lignes = []

    page = pdf.pages[13]
    tables = page.extract_tables()
    for tbl in tables:
        if not tbl:
            continue
        for row in tbl:
            if not row or not row[0]:
                continue
            rc = [_clean(str(c)) if c else "" for c in row]
            label = rc[0]
            if label in ("Parts A", "Parts A'", "Parts B"):
                nums = [_num_or_none(v) for v in rc[1:] if v]
                lignes.append({
                    "type_parts": label,
                    "nombre_parts":                    nums[0] if len(nums) > 0 else None,
                    "remboursement_montant_libere":     nums[1] if len(nums) > 1 else None,
                    "affectation_revenu_prioritaire":   nums[2] if len(nums) > 2 else None,
                    "actif_net":                        nums[3] if len(nums) > 3 else None,
                    "valeur_liquidative":               nums[4] if len(nums) > 4 else None,
                })

    return {titre: lignes}


# ── VIII. Exposition portefeuille (pages 16-18) ─────────────
def _exposition_portefeuille(pdf):
    """
    Clé: "VIII. Exposition portefeuille"
    Structure: {
        "8.1 - Ventilation entre actifs de capital investissement et autres actifs éligibles": [
            {"libelle", "actifs_capital_investissement", "autres_actifs", "total_bilan"}, ...
        ],
        "8.2 - Décomposition du portefeuille de capital investissement et évolution des investissements": [
            {"nom", "devise", "cout_acq_N", "cout_acq_N1", "cout_acq_variation",
             "eval_N", "eval_N1", "eval_variation"}, ...
        ]
    }
    """
    result = {
        "8.1 - Ventilation entre actifs de capital investissement et autres actifs éligibles": [],
        "8.2 - Décomposition du portefeuille de capital investissement et évolution des investissements": [],
    }

    # 8.1 — page 16 (index 15)
    page_81 = pdf.pages[15]
    tables_81 = page_81.extract_tables()
    for tbl in tables_81:
        if not tbl:
            continue
        for row in tbl:
            if not row or not row[0]:
                continue
            rc = [_clean(str(c)) if c else "" for c in row]
            label = rc[0]
            if label in ("", "Ventilation entre actifs de capital investissement et autres actifs éligibles",
                         "Total", "Actions", "Obligations convertibles", "Autres obligations"):
                if label == "Total":
                    nums = [_num_or_none(v) for v in rc[1:] if v]
                    result["8.1 - Ventilation entre actifs de capital investissement et autres actifs éligibles"].append({
                        "libelle": "Total",
                        "actifs_capital_investissement": nums[0] if nums else None,
                        "autres_actifs": nums[1] if len(nums) > 1 else None,
                        "total_bilan": nums[2] if len(nums) > 2 else None,
                    })
                continue
            nums = [_num_or_none(v) for v in rc[1:] if v]
            result["8.1 - Ventilation entre actifs de capital investissement et autres actifs éligibles"].append({
                "libelle": label,
                "actifs_capital_investissement": nums[0] if len(nums) > 0 else None,
                "autres_actifs": nums[1] if len(nums) > 1 else None,
                "total_bilan": nums[2] if len(nums) > 2 else None,
            })

    # 8.2 — pages 17-18 (index 16, 17)
    seen = set()
    for pi in [16, 17]:
        page = pdf.pages[pi]
        entries = _split_multirow_table(page)
        for e in entries:
            if e.get("_total"):
                continue
            nom = e["nom"]
            if nom in seen:
                continue
            seen.add(nom)
            vs = e["values"]
            result["8.2 - Décomposition du portefeuille de capital investissement et évolution des investissements"].append({
                "nom": nom,
                "devise": vs[0] if len(vs) > 0 else "Euro",
                "cout_acq_N":        _num_or_none(vs[1]) if len(vs) > 1 else None,
                "cout_acq_N1":       _num_or_none(vs[2]) if len(vs) > 2 else None,
                "cout_acq_variation":_num_or_none(vs[3]) if len(vs) > 3 else None,
                "eval_N":            _num_or_none(vs[4]) if len(vs) > 4 else None,
                "eval_N1":           _num_or_none(vs[5]) if len(vs) > 5 else None,
                "eval_variation":    _num_or_none(vs[6]) if len(vs) > 6 else None,
            })

    return result


# ── Cessions (page 21) ──────────────────────────────────────
def _cessions(pdf):
    """
    Clé: "Cessions"
    Structure: {
        "8.4 - Etat des cessions et des sorties d'actifs de l'exercice": [
            {"nom", "cout_acquisition", "prix_cession", "plus_values", "moins_values"}, ...
        ]
    }
    """
    titre = "8.4 - Etat des cessions et des sorties d'actifs de l'exercice"
    lignes = []

    page = pdf.pages[20]
    tables = page.extract_tables()
    for tbl in tables:
        if not tbl or len(tbl) < 2:
            continue
        # Chercher la ligne d'en-tête
        header_idx = None
        for i, row in enumerate(tbl):
            if row and any("Prix de cession" in str(c) or "acquisition" in str(c) for c in row if c):
                header_idx = i
                break
        if header_idx is None:
            continue

        for row in tbl[header_idx + 1:]:
            if not row or not row[0]:
                continue
            cell0 = str(row[0])
            # Cellule 0 peut contenir plusieurs noms séparés par \n
            noms = [n.strip() for n in cell0.split("\n") if n.strip()]
            # Les autres colonnes contiennent les valeurs dans le même ordre
            cols_vals = []
            for c in row[1:]:
                if c is None:
                    cols_vals.append([])
                else:
                    vals = [v.strip() for v in str(c).split("\n") if v.strip()]
                    cols_vals.append(vals)

            for idx, nom in enumerate(noms):
                if nom in ("Total", "Nom de la société et nature des instruments"):
                    continue
                cout  = _num_or_none(cols_vals[0][idx]) if 0 < len(cols_vals) and idx < len(cols_vals[0]) else None
                prix  = _num_or_none(cols_vals[1][idx]) if 1 < len(cols_vals) and idx < len(cols_vals[1]) else None
                pv    = _num_or_none(cols_vals[2][idx]) if 2 < len(cols_vals) and idx < len(cols_vals[2]) else None
                mv    = _num_or_none(cols_vals[3][idx]) if 3 < len(cols_vals) and idx < len(cols_vals[3]) else None
                lignes.append({
                    "nom": nom,
                    "cout_acquisition": cout,
                    "prix_cession": prix,
                    "plus_values": pv,
                    "moins_values": mv,
                })

    return {titre: lignes}


# ── Créances et frais (page 22) ─────────────────────────────
def _creances_et_frais(pdf):
    """
    Clé: "Créances et frais"
    Structure: {
        "IX. Créances et dettes : ventilation par nature": {
            "creances": [{"libelle", "montant"}, ...],
            "total_creances": float,
            "dettes": [{"libelle", "montant"}, ...],
            "total_dettes": float
        },
        "X. Frais de gestion, autres frais et charges": {
            "lignes": [{"libelle", "montant"}, ...],
            "total_frais_gestion_societe": float,
            "total_frais_fonctionnement": float,
            "pct_actif_net": float,
            "commission_surperformance": float,
            "total_general": float
        }
    }
    """
    res_IX = {"creances": [], "total_creances": None, "dettes": [], "total_dettes": None}
    res_X = {
        "lignes": [],
        "total_frais_gestion_societe": None,
        "total_frais_fonctionnement": None,
        "pct_actif_net": None,
        "commission_surperformance": None,
        "total_general": None,
    }

    page = pdf.pages[21]
    tables = page.extract_tables()

    def _parse_multirow_2col(tbl, section_name):
        """Parse une table 2 colonnes avec cellules multirow."""
        entries = []
        total = None
        for row in tbl[1:]:  # skip header
            if not row or not row[0]:
                continue
            cell_label = str(row[0])
            cell_val = str(row[1]) if len(row) > 1 and row[1] else ""

            labels = [l.strip() for l in cell_label.split("\n") if l.strip()]
            vals = [v.strip() for v in cell_val.split("\n") if v.strip()]

            for i, label in enumerate(labels):
                if label == "Total":
                    # Le total est dans la ligne suivante ou dans les vals
                    pass
                elif label:
                    montant = _num_or_none(vals[i]) if i < len(vals) else None
                    entries.append({"libelle": label, "montant": montant})

            # Vérifier si c'est la ligne Total dans la prochaine itération
        # Chercher le total dans la dernière ligne
        for row in tbl:
            if row and row[0] and str(row[0]).strip() == "Total":
                cell_val = str(row[1]) if len(row) > 1 and row[1] else ""
                total = _num_or_none(cell_val.split("\n")[0].strip() if "\n" in cell_val else cell_val)
        return entries, total

    for tbl in tables:
        if not tbl or not tbl[0]:
            continue
        header = str(tbl[0][0]).strip() if tbl[0][0] else ""

        if header == "Créances":
            entries, total = _parse_multirow_2col(tbl, "creances")
            res_IX["creances"] = entries
            res_IX["total_creances"] = total

        elif header == "Dettes":
            entries, total = _parse_multirow_2col(tbl, "dettes")
            res_IX["dettes"] = entries
            res_IX["total_dettes"] = total

        else:
            # Tableau frais (X) : colonnes label | montant
            for row in tbl[1:]:
                if not row or not row[0]:
                    continue
                cell_label = str(row[0])
                cell_val = str(row[1]) if len(row) > 1 and row[1] else ""

                labels = [l.strip() for l in cell_label.split("\n") if l.strip()]
                vals = [v.strip() for v in cell_val.split("\n") if v.strip()]

                for i, label in enumerate(labels):
                    if not label or label.startswith("*") or label.startswith("("):
                        continue
                    montant_raw = vals[i] if i < len(vals) else ""
                    # Convertir "1,28%" en float
                    if "%" in montant_raw:
                        try:
                            res_X["pct_actif_net"] = float(montant_raw.replace("%", "").replace(",", ".").strip()) / 100
                        except ValueError:
                            pass
                        continue
                    montant = _num_or_none(montant_raw)
                    if "Total frais de gestion de la société" in label:
                        res_X["total_frais_gestion_societe"] = montant
                    elif "Total frais de fonctionnement" in label:
                        res_X["total_frais_fonctionnement"] = montant
                    elif "Commission de surperformance" in label:
                        res_X["commission_surperformance"] = montant
                    elif "Total frais de gestion autres" in label:
                        res_X["total_general"] = montant
                    elif montant is not None:
                        res_X["lignes"].append({"libelle": label, "montant": montant})

    return {
        "IX. Créances et dettes : ventilation par nature": res_IX,
        "X. Frais de gestion, autres frais et charges": res_X,
    }


# ── Hors-bilan et autres infos (page 23) ────────────────────
def _hors_bilan(pdf):
    """
    Clé: "Hors-bilan et autres infos"
    Structure: {
        "XI. Engagements de hors-bilan reçus et donnés": {
            "lignes": [{"libelle", "valeur"}, ...],
            "total": float,
            "detail_titres": str
        },
        "XII. Autres informations": {
            "valeurs": {"libelle": "valeur/texte", ...},
            "titres_entites_liees": {"Actions": val, "Obligations": val, ...}
        }
    }
    """
    res_XI = {"lignes": [], "total": None, "detail_titres": ""}
    res_XII = {"valeurs": {}, "titres_entites_liees": {}}

    page = pdf.pages[22]
    tables = page.extract_tables()
    mode = None

    for tbl in tables:
        if not tbl:
            continue
        for row in tbl:
            if not row:
                continue
            rc = [_clean(str(c)) if c else "" for c in row]
            label = rc[0]
            if "XI." in label:
                mode = "XI"; continue
            if "XII." in label:
                mode = "XII"; continue
            if not label:
                continue

            val_raw = next((v for v in rc[1:] if v), "")
            val = _num_or_none(val_raw)

            if mode == "XI":
                if "Les titres" in label or "Menwell" in label or label.startswith("-"):
                    res_XI["detail_titres"] += label + " "
                elif label == "Total":
                    res_XI["total"] = val
                else:
                    res_XI["lignes"].append({"libelle": label, "valeur": val})
            elif mode == "XII":
                if label in ("Actions", "Obligations", "TCN", "OPC",
                             "Avances en compte-courant", "Total"):
                    res_XII["titres_entites_liees"][label] = val
                else:
                    res_XII["valeurs"][label] = val_raw if val is None else val

    return {
        "XI. Engagements de hors-bilan reçus et donnés": res_XI,
        "XII. Autres informations": res_XII,
    }


# ── XIII. Sommes distribuables (page 24) ────────────────────
def _sommes_distribuables(pdf):
    """
    Clé: "XIII. Sommes distribuables"
    Structure: {
        "note": "En l'absence de...",
        "Affectation des sommes distribuables afférentes aux revenus nets": {
            "libelle": {"N": val, "N1": val}, ...
        },
        "Affectation des sommes distribuables afférentes aux plus et moins-values réalisées nettes": {
            "libelle": {"N": val, "N1": val}, ...
        }
    }
    """
    result = {
        "note": "",
        "Affectation des sommes distribuables afférentes aux revenus nets": {},
        "Affectation des sommes distribuables afférentes aux plus et moins-values réalisées nettes": {},
    }

    rows = _extract_two_col_table([23], pdf)
    mode = None
    for r in rows:
        label = r["libelle"]
        if "En l'absence" in label:
            result["note"] = label
            continue
        if "afférentes aux revenus nets" in label:
            mode = "revenus"; continue
        if "afférentes aux plus et moins-values réalisées nettes" in label:
            mode = "pmv"; continue
        if not label or label.startswith("XIII") or label.startswith("SINGULAR") or label.startswith("Page"):
            continue
        if mode == "revenus":
            result["Affectation des sommes distribuables afférentes aux revenus nets"][label] = {
                "N": r["N"], "N1": r["N1"]
            }
        elif mode == "pmv":
            result["Affectation des sommes distribuables afférentes aux plus et moins-values réalisées nettes"][label] = {
                "N": r["N"], "N1": r["N1"]
            }

    return result


# ── XIV. Elements de bilan (pages 25-26) ────────────────────
def _elements_de_bilan(pdf):
    """
    Clé: "XIV. Elements de bilan"
    Structure: {
        "14.1 - Inventaire des dépôts et des instruments financiers": [
            {"nom", "quantite", "valeur_actuelle_euros", "devise",
             "pct_actif_net", "secteur_activite"}, ...
        ],
        "total_quantite": float,
        "total_valeur": float
    }
    """
    titre = "14.1 - Inventaire des dépôts et des instruments financiers"
    instruments = []
    total_qte = total_val = None
    seen = set()

    for pi in [24, 25]:  # pages 25 et 26 (index 24, 25)
        page = pdf.pages[pi]
        entries = _split_multirow_table(page)
        for e in entries:
            if e.get("_total"):
                vs = e["values"]
                nums = [_num_or_none(v) for v in vs if v]
                if nums:
                    total_qte = nums[0]
                    if len(nums) > 1:
                        total_val = nums[1]
                continue
            nom = e["nom"]
            if nom in seen:
                continue
            seen.add(nom)
            vs = e["values"]
            instruments.append({
                "nom": nom,
                "quantite":             _num_or_none(vs[0]) if len(vs) > 0 else None,
                "valeur_actuelle_euros":_num_or_none(vs[1]) if len(vs) > 1 else None,
                "devise":               vs[2] if len(vs) > 2 else "Euro",
                "pct_actif_net":        _num_or_none(vs[3]) if len(vs) > 3 else None,
                "secteur_activite":     " ".join(vs[4:]).strip() if len(vs) > 4 else "",
            })

    return {
        titre: instruments,
        "total_quantite": total_qte,
        "total_valeur": total_val,
    }


# ══════════════════════════════════════════════════════════════
#  POINT D'ENTRÉE PRINCIPAL
# ══════════════════════════════════════════════════════════════

def extract(pdf_path: str) -> dict:
    """
    Extrait le dictionnaire complet depuis le PDF.

    Les clés sont STATIQUES, identiques aux sheet names Excel Coala :
        "Bilan Actif", "Bilan Passif", "Compte de résultat 1",
        "Compte de résultat 2", "I. Prospectus", "I. Caractéristiques",
        "II. Copie règlement 1", "II. Copie règlement 2",
        "III. Capitaux propres", "IV. Evol parts", "V. Flux nominal",
        "VI. Ventilation AN 1", "VI. Ventilation AN 2",
        "VIII. Exposition portefeuille", "Cessions",
        "Créances et frais", "Hors-bilan et autres infos",
        "XIII. Sommes distribuables", "XIV. Elements de bilan"

    Args:
        pdf_path: Chemin vers le PDF du compte annuel.

    Returns:
        dict avec les clés statiques ci-dessus.
    """
    print(f"[extract] {pdf_path}")

    # Ouvrir le PDF une seule fois et le passer aux parseurs
    with pdfplumber.open(pdf_path) as pdf:
        steps = [
            ("Bilan Actif",                   lambda: _bilan_actif(pdf)),
            ("Bilan Passif",                  lambda: _bilan_passif(pdf)),
            ("Compte de résultat 1",          lambda: _compte_resultat_1(pdf)),
            ("Compte de résultat 2",          lambda: _compte_resultat_2(pdf)),
            ("I. Prospectus",                 lambda: _prospectus(pdf)),
            ("I. Caractéristiques",           lambda: _caracteristiques(pdf)),
            ("II. Copie règlement 1",         lambda: _regles_1(pdf)),
            ("II. Copie règlement 2",         lambda: _regles_2(pdf)),
            ("III. Capitaux propres",         lambda: _capitaux_propres_wrap(pdf, pdf_path)),
            ("IV. Evol parts",               lambda: _evol_parts(pdf)),
            ("V. Flux nominal",              lambda: _flux_nominal(pdf)),
            ("VI. Ventilation AN 1",         lambda: _ventilation_AN1(pdf)),
            ("VI. Ventilation AN 2",         lambda: _ventilation_AN2(pdf)),
            ("VIII. Exposition portefeuille",lambda: _exposition_portefeuille(pdf)),
            ("Cessions",                     lambda: _cessions(pdf)),
            ("Créances et frais",            lambda: _creances_et_frais(pdf)),
            ("Hors-bilan et autres infos",   lambda: _hors_bilan(pdf)),
            ("XIII. Sommes distribuables",   lambda: _sommes_distribuables(pdf)),
            ("XIV. Elements de bilan",       lambda: _elements_de_bilan(pdf)),
        ]

        result = {}
        for key, fn in steps:
            print(f"  {key}...", end=" ", flush=True)
            try:
                result[key] = fn()
                print("✓")
            except Exception as e:
                print(f"✗ ({e})")
                result[key] = {}

    print(f"[extract] ✓ {len(result)} sections")
    return result


def _capitaux_propres_wrap(pdf, pdf_path):
    """
    Page 10 (index 9). Positions spatiales exactes :
        signe    x ≈ 290-310
        Cumul N  x ≈ 315-375
        Cumul N1 x ≈ 385-445
        Variation x ≈ 465-520
    """
    titre = 'III. Reconstitution de la ligne "capitaux propres"'
    result = {titre: {"lignes": [], "notes": {}}}
    inner = result[titre]

    page = pdf.pages[9]
    words = page.extract_words()
    by_y = _words_to_rows(words)

    col_signe = (285, 315)
    col_cumN  = (315, 380)
    col_cumN1 = (380, 460)
    col_var   = (460, 525)

    _SKIP = {
        "Reconstitution de la ligne des capitaux propres depuis",
        "l'origine Cumul Exercice N Cumul Exercice N-1 Variation Exercice N",
        "Cumul Exercice N", "Cumul Exercice N-1", "Variation Exercice N",
        "l'origine", "SINGULAR VENTURES I FPCI - Comptes annuels au 31/12/2025",
    }

    for y, row_words in by_y.items():
        label_parts, cumN_parts, cumN1_parts, var_parts, signe = [], [], [], [], None

        for w in row_words:
            x, t = w["x0"], w["text"]
            if col_signe[0] <= x < col_signe[1] and t in ("+", "-", "+/-", "="):
                signe = t
            elif col_cumN[0] <= x < col_cumN[1]:
                cumN_parts.append(t)
            elif col_cumN1[0] <= x < col_cumN1[1]:
                cumN1_parts.append(t)
            elif col_var[0] <= x < col_var[1]:
                var_parts.append(t)
            elif t not in ("+", "-", "+/-", "="):
                label_parts.append(t)

        label = " ".join(label_parts).strip()
        if not label:
            continue
        if label.startswith("SINGULAR") or label.startswith("Page") or label.startswith("III."):
            continue
        if any(skip in label for skip in _SKIP):
            continue

        # Notes bas de page
        if "Frais de constitution" in label:
            m = re.search(r"([-\d\s]+)\s*€", label)
            if m:
                inner["notes"]["frais_constitution"] = _num(m.group(1))
            continue
        if "Prime de souscription" in label:
            m = re.search(r"([\d\s]+)\s*€", label)
            if m:
                inner["notes"]["prime_souscription"] = _num(m.group(1))
            continue
        # Footnotes
        if label.startswith("("):
            continue

        cumN  = _num_or_none(" ".join(cumN_parts))
        cumN1 = _num_or_none(" ".join(cumN1_parts))
        var   = _num_or_none(" ".join(var_parts))

        inner["lignes"].append({
            "libelle": label,
            "signe": signe,
            "cumul_N": cumN,
            "cumul_N1": cumN1,
            "variation_N": var,
        })

    return result


# ─── CLI ──────────────────────────────────────────────────
if __name__ == "__main__":
    import sys, json

    def _serial(o):
        if hasattr(o, "isoformat"):
            return o.isoformat()
        return str(o)

    path = sys.argv[1] if len(sys.argv) > 1 else "comptes_annuels.pdf"
    d = extract(path_pdf_file)
    print(json.dumps(d, ensure_ascii=False, indent=2, default=_serial))

[extract] C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Singular Ventures I - Comptes annuels 31.12.2025 v3.pdf
  Bilan Actif... ✓
  Bilan Passif... ✓
  Compte de résultat 1... ✓
  Compte de résultat 2... ✓
  I. Prospectus... ✓
  I. Caractéristiques... ✓
  II. Copie règlement 1... ✓
  II. Copie règlement 2... ✓
  III. Capitaux propres... ✓
  IV. Evol parts... ✓
  V. Flux nominal... ✓
  VI. Ventilation AN 1... ✓
  VI. Ventilation AN 2... ✓
  VIII. Exposition portefeuille... ✓
  Cessions... ✓
  Créances et frais... ✓
  Hors-bilan et autres infos... ✓
  XIII. Sommes distribuables... ✓
  XIV. Elements de bilan... ✓
[extract] ✓ 19 sections
{
  "Bilan Actif": [
    {
      "libelle": "Bilan Actif au 31/12/2025 en euros",
      "niveau": 0,
      "N": null,
      "N1": null
    },
    {
      "libelle": "IMMOBILISATIONS CORPORELLES NETTES",
      "niveau": 0,
      "N": null,
      "N1": null
    },
    {
      "libelle": "TITRES FINANCIERS",
      "niveau": 